# Predicting asset failure within 90 days

This experiment asks whether information available **at inspection time** can rank assets by their risk of failing within 90 days. It uses chronological validation, because a random split would give an unrealistically easy estimate when failure prevalence changes over time.

Primary metric: average precision (AP), compared with the failure prevalence baseline. Secondary metrics: ROC AUC, Brier score, and recall/lift when inspecting the riskiest 10% of records.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             brier_score_loss, PrecisionRecallDisplay)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
pd.set_option('display.max_colwidth', 120)
warnings.filterwarnings('ignore', category=FutureWarning)


In [ ]:
data_path, dictionary_path = Path('data.csv'), Path('columns.csv')
assert data_path.exists() and dictionary_path.exists(), 'Run this notebook beside data.csv and columns.csv'
dictionary = pd.read_csv(dictionary_path)
df = pd.read_csv(data_path)
df['inspected_at'] = pd.to_datetime(df['inspected_at'], errors='raise')
df = df.sort_values('inspected_at').reset_index(drop=True)
display(dictionary)
print(f'{len(df):,} rows, {df.shape[1]} columns; {df.inspected_at.min()} through {df.inspected_at.max()}')
print(f"Failures: {df.failed_90d.sum():,} ({df.failed_90d.mean():.1%}); duplicate rows: {df.duplicated().sum()}; missing cells: {df.isna().sum().sum()}")
assert set(df.failed_90d.unique()) <= {0, 1}
assert df.asset_tag.nunique() == len(df), 'Repeated assets require grouped validation to avoid identity leakage.'


## Leakage and data quality

`asset_tag` is an identifier and is unique here, so it cannot generalize. `parts_replaced` is documented as the number of parts on the work order that **closed** the inspection record. That is downstream information and may not exist when a prediction must be made. We therefore exclude both from the deployable feature set. The table below quantifies why this matters.

In [ ]:
leakage_table = df.groupby('parts_replaced')['failed_90d'].agg(records='size', failures='sum', failure_rate='mean')
display(leakage_table.style.format({'failure_rate': '{:.1%}'}))
quarterly = df.set_index('inspected_at')['failed_90d'].resample('QE').agg(['size', 'mean'])
display(quarterly.rename(columns={'size': 'records', 'mean': 'failure_rate'}).style.format({'failure_rate': '{:.1%}'}))
ax = quarterly['mean'].plot(marker='o', title='Failure prevalence changes substantially over time',
                            ylabel='Failure rate', figsize=(8, 3))
ax.axhline(df.failed_90d.mean(), color='grey', ls='--', label='overall')
ax.legend(); plt.tight_layout(); plt.show()


The changing base rate makes chronology important. It also means probability calibration may decay even if risk ranking remains useful.

In [ ]:
# Only transformations available from the inspection timestamp itself.
df['inspection_month'] = df.inspected_at.dt.month
df['inspection_dow'] = df.inspected_at.dt.dayofweek
df['inspection_hour'] = df.inspected_at.dt.hour
features = ['site', 'vibration_rms', 'bearing_temp_c', 'ambient_temp_c',
            'meter_hours', 'load_pct', 'inspection_month', 'inspection_dow', 'inspection_hour']
numeric = [c for c in features if c != 'site']
categorical = ['site']

linear_prep = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
])
tree_prep = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
], remainder='passthrough')
candidates = {
    'logistic': Pipeline([('prep', linear_prep), ('model', LogisticRegression(max_iter=2000))]),
    'extra_trees_leaf10': Pipeline([('prep', tree_prep), ('model', ExtraTreesClassifier(
        n_estimators=400, min_samples_leaf=10, max_features=0.8, n_jobs=-1, random_state=SEED))]),
    'extra_trees_leaf40': Pipeline([('prep', tree_prep), ('model', ExtraTreesClassifier(
        n_estimators=400, min_samples_leaf=40, max_features=0.8, n_jobs=-1, random_state=SEED))]),
}

cut = int(0.80 * len(df))
development, holdout = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(f'Development: {development.inspected_at.min()} to {development.inspected_at.max()} ({len(development):,} rows)')
print(f'Untouched holdout: {holdout.inspected_at.min()} to {holdout.inspected_at.max()} ({len(holdout):,} rows, {holdout.failed_90d.sum()} failures)')


## Model selection on rolling-origin splits

Each validation fold occurs strictly after its training data. The final holdout is not used for model selection.

In [ ]:
rows = []
tscv = TimeSeriesSplit(n_splits=4)
X_dev, y_dev = development[features], development.failed_90d
for fold, (train_idx, valid_idx) in enumerate(tscv.split(development), 1):
    y_train, y_valid = y_dev.iloc[train_idx], y_dev.iloc[valid_idx]
    for name, estimator in candidates.items():
        model = clone(estimator).fit(X_dev.iloc[train_idx], y_train)
        probability = model.predict_proba(X_dev.iloc[valid_idx])[:, 1]
        rows.append({
            'model': name, 'fold': fold,
            'validation_start': development.inspected_at.iloc[valid_idx[0]],
            'prevalence': y_valid.mean(),
            'AP': average_precision_score(y_valid, probability),
            'ROC_AUC': roc_auc_score(y_valid, probability),
        })
cv = pd.DataFrame(rows)
display(cv.round(4))
cv_summary = cv.groupby('model').agg(
    AP_mean=('AP', 'mean'), AP_std=('AP', 'std'),
    ROC_AUC_mean=('ROC_AUC', 'mean'), folds=('fold', 'size')
).sort_values('AP_mean', ascending=False)
display(cv_summary.round(4))
winner_name = cv_summary.index[0]
print('Selected by mean rolling-validation AP:', winner_name)


In [ ]:
# One final evaluation on the latest 20%.
winner = clone(candidates[winner_name]).fit(development[features], development.failed_90d)
y_test = holdout.failed_90d.to_numpy()
p_test = winner.predict_proba(holdout[features])[:, 1]
k = max(1, int(np.ceil(0.10 * len(holdout))))
top = np.argsort(-p_test)[:k]
metrics = {
    'holdout_rows': len(holdout),
    'holdout_failures': int(y_test.sum()),
    'prevalence / random AP': y_test.mean(),
    'average_precision': average_precision_score(y_test, p_test),
    'ROC_AUC': roc_auc_score(y_test, p_test),
    'Brier_score (lower is better)': brier_score_loss(y_test, p_test),
    'recall_at_top_10pct': y_test[top].sum() / y_test.sum(),
    'precision_at_top_10pct': y_test[top].mean(),
    'lift_at_top_10pct': y_test[top].mean() / y_test.mean(),
}
display(pd.Series(metrics, name='latest holdout').to_frame().round(4))
PrecisionRecallDisplay.from_predictions(y_test, p_test, name=winner_name)
plt.axhline(y_test.mean(), color='grey', ls='--', label='random baseline')
plt.legend(); plt.title('Precision-recall on latest chronological holdout'); plt.show()


## Uncertainty and model interpretation

The holdout contains relatively few failures, so the bootstrap interval is intentionally shown. It measures sampling uncertainty, not future operational drift.

In [ ]:
rng = np.random.default_rng(SEED)
boot = []
for _ in range(1000):
    idx = rng.integers(0, len(y_test), len(y_test))
    if np.unique(y_test[idx]).size == 2:
        boot.append((average_precision_score(y_test[idx], p_test[idx]),
                     roc_auc_score(y_test[idx], p_test[idx])))
boot = np.asarray(boot)
ci = pd.DataFrame({
    'estimate': [metrics['average_precision'], metrics['ROC_AUC']],
    'bootstrap_2.5%': np.quantile(boot, .025, axis=0),
    'bootstrap_97.5%': np.quantile(boot, .975, axis=0),
}, index=['AP', 'ROC AUC'])
display(ci.round(4))
perm = permutation_importance(
    winner, holdout[features], y_test, scoring='average_precision',
    n_repeats=20, random_state=SEED, n_jobs=-1
)
importance = pd.Series(perm.importances_mean, index=features).sort_values()
importance.plot.barh(figsize=(7, 4), title='Permutation importance on holdout (decrease in AP)')
plt.xlabel('Mean AP decrease'); plt.tight_layout(); plt.show()


## Leakage sensitivity check

This is diagnostic only. If adding `parts_replaced` causes a dramatic jump, that supports treating it as post-inspection leakage rather than deployable signal.

In [ ]:
leaky_features = features + ['parts_replaced']
leaky_prep = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
], remainder='passthrough')
leaky_model = Pipeline([('prep', leaky_prep), ('model', ExtraTreesClassifier(
    n_estimators=400, min_samples_leaf=10, max_features=.8, n_jobs=-1, random_state=SEED))])
leaky_model.fit(development[leaky_features], development.failed_90d)
p_leaky = leaky_model.predict_proba(holdout[leaky_features])[:, 1]
display(pd.DataFrame({
    'deployable_features': [metrics['average_precision'], metrics['ROC_AUC']],
    'including_parts_replaced (invalid)': [
        average_precision_score(y_test, p_leaky), roc_auc_score(y_test, p_leaky)
    ],
}, index=['AP', 'ROC AUC']).round(4))


In [ ]:
ap = metrics['average_precision']
baseline = metrics['prevalence / random AP']
lift = metrics['lift_at_top_10pct']
signal = 'useful ranking signal' if ap >= 2 * baseline and lift >= 2 else 'not yet convincing signal'
verdict = (
    f"**Verdict:** There is {signal} on the latest holdout "
    f"(AP {ap:.3f} versus random {baseline:.3f}; top-decile lift {lift:.1f}×). "
    "This supports a monitored prioritization pilot, not autonomous maintenance decisions. "
    "The latest holdout has few failures, the base rate shifts sharply over time, and the apparent "
    "near-perfect signal from parts_replaced is leakage. Before deployment, confirm feature availability "
    "at scoring time, collect more recent outcomes, retrain on a schedule, and monitor AP, top-decile "
    "recall, and calibration by time period/site."
)
display(Markdown(verdict))
